# 01 - Case Base Preparation

This notebook processes PDF case files for the CBR (Case-Based Reasoning) Pemalsuan Surat (Document Forgery) system.

## Pipeline
1. **Read PDFs** from `data/raw_pdf`
2. **Extract Text** using pdfplumber
3. **Clean Text**: lowercase, remove page numbers, watermark, header, footer, punctuation
4. **Tokenize** and **Remove Stopwords**
5. **Stem** using Sastrawi (Indonesian stemmer)
6. **Save** cleaned text to `data/raw_text`
7. **Generate** cleaning log

In [ ]:
# Imports
import os
import re
import pdfplumber
from pathlib import Path
import pandas as pd
from datetime import datetime
from tqdm import tqdm

# NLP Libraries
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Indonesian Stemmer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

In [ ]:
# Paths
PROJECT_ROOT = Path("../..")
RAW_PDF_DIR = PROJECT_ROOT / "data" / "raw_pdf"
RAW_TEXT_DIR = PROJECT_ROOT / "data" / "raw_text"
LOG_DIR = PROJECT_ROOT / "data" / "results"

# Create directories if not exist
RAW_TEXT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {PROJECT_ROOT}")
print(f"PDF Directory: {RAW_PDF_DIR}")
print(f"Output Directory: {RAW_TEXT_DIR}")

In [ ]:
class TextCleaner:
    """Text cleaning utilities for Indonesian legal documents."""

    def __init__(self):
        self.stemmer = StemmerFactory().createStemmer()
        self.stop_words = set(stopwords.words('indonesian'))
        
    def clean(self, text):
        """Main cleaning pipeline."""
        # 1. Merge text (already done by pdfplumber)
        text = self._lowercase(text)
        text = self._remove_page_number(text)
        text = self._remove_watermark(text)
        text = self._remove_header_footer(text)
        text = self._remove_punctuation(text)
        tokens = self._tokenize(text)
        tokens = self._remove_stopwords(tokens)
        tokens = self._stem(tokens)
        return " ".join(tokens)

    def _lowercase(self, text):
        return text.lower()

    def _remove_page_number(self, text):
        # Remove common page number patterns
        patterns = [
            r'\bpage\s*\d+\b',
            r'\bhalaman\s*\d+\b',
            r'\b\d+\s*/\s*\d+\b',  # e.g., 1/15
            r'^\s*\d+\s*$',  # standalone numbers
        ]
        for pattern in patterns:
            text = re.sub(pattern, '', text, flags=re.MULTILINE | re.IGNORECASE)
        return text

    def _remove_watermark(self, text):
        # Common watermark patterns
        watermarks = [
            r'\bsecret\b',
            r'\bconfidential\b',
            r'\bdraft\b',
            r'\bcopy\b',
            r'\bdokumen\s*tidak\s*berlaku\b',
        ]
        for wm in watermarks:
            text = re.sub(wm, '', text, flags=re.IGNORECASE)
        return text

    def _remove_header_footer(self, text):
        # Remove repeated header/footer patterns
        # Usually short lines at start or end of pages
        lines = text.split('\n')
        if len(lines) > 2:
            # Remove first line if short (potential header)
            if len(lines[0].strip()) < 50:
                lines = lines[1:]
            # Remove last line if short (potential footer)
            if len(lines[-1].strip()) < 50:
                lines = lines[:-1]
        return '\n'.join(lines)

    def _remove_punctuation(self, text):
        # Keep Indonesian characters (a-z) and numbers
        text = re.sub(r'[^a-z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)  # normalize whitespace
        return text.strip()

    def _tokenize(self, text):
        return word_tokenize(text)

    def _remove_stopwords(self, tokens):
        return [t for t in tokens if t not in self.stop_words and len(t) > 1]

    def _stem(self, tokens):
        return [self.stemmer.stem(t) for t in tokens]


cleaner = TextCleaner()
print("TextCleaner initialized with Sastrawi stemmer")

In [ ]:
def extract_text_from_pdf(pdf_path):
    """Extract all text from PDF using pdfplumber."""
    text_pages = []
    
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for i, page in enumerate(pdf.pages):
                page_text = page.extract_text()
                if page_text:
                    text_pages.append(page_text)
    except Exception as e:
        print(f"Error reading {pdf_path.name}: {e}")
        return None

    return "\n".join(text_pages)


# Test with one PDF
test_pdf = list(RAW_PDF_DIR.glob("*.pdf"))[0]
test_text = extract_text_from_pdf(test_pdf)
print(f"Sample PDF: {test_pdf.name}")
print(f"Extracted characters: {len(test_text) if test_text else 0}")
print("-" * 50)
print(test_text[:500] if test_text else "No text extracted")

In [ ]:
# Process all PDFs
pdf_files = sorted(RAW_PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files")

logs = []

for pdf_file in tqdm(pdf_files, desc="Processing PDFs"):
    # Extract
    raw_text = extract_text_from_pdf(pdf_file)
    
    if raw_text is None:
        logs.append({
            "filename": pdf_file.name,
            "status": "ERROR - Extraction Failed",
            "raw_chars": 0,
            "cleaned_tokens": 0
        })
        continue

    raw_char_count = len(raw_text)
    
    # Clean
    cleaned_text = cleaner.clean(raw_text)
    
    # Save
    output_filename = pdf_file.stem + ".txt"
    output_path = RAW_TEXT_DIR / output_filename
    output_path.write_text(cleaned_text, encoding='utf-8')

    cleaned_token_count = len(cleaned_text.split())
    
    logs.append({
        "filename": pdf_file.name,
        "status": "SUCCESS",
        "raw_chars": raw_char_count,
        "cleaned_tokens": cleaned_token_count
    })

print(f"\nProcessed {len(pdf_files)} files")

In [ ]:
# Save cleaning log
log_df = pd.DataFrame(logs)
log_filename = f"cleaning_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
log_path = LOG_DIR / log_filename
log_df.to_csv(log_path, index=False)

print(f"Cleaning log saved to: {log_path}")
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Total PDFs processed: {len(logs)}")
print(f"Successful: {len([l for l in logs if l['status'] == 'SUCCESS'])}")
print(f"Failed: {len([l for l in logs if l['status'].startswith('ERROR')])}")
print("\nLog DataFrame:")
log_df

In [ ]:
# Verify output
output_files = list(RAW_TEXT_DIR.glob("*.txt"))
print(f"Output files created: {len(output_files)}")

# Show sample cleaned text
if output_files:
    sample_file = output_files[0]
    print(f"\nSample cleaned file: {sample_file.name}")
    print("-" * 50)
    print(sample_file.read_text(encoding='utf-8')[:300])